# iRIS Replication — Setup Verifier

Run each cell top-to-bottom after `conda activate iris`.
Select kernel: **Python (iris)**.

In [ ]:
import sys, platform, importlib
print("Python:", sys.version)
print("Executable:", sys.executable)
print("Platform:", platform.platform())
assert sys.version_info[:2] == (3, 10), "Need Python 3.10.x"

In [ ]:
REQUIRED = {"torch":"2.3.1","torchvision":"0.18.1","numpy":"1.26.4",
            "scipy":"1.13.1","matplotlib":"3.8.4","yaml":"6.0.1",
            "tqdm":"4.66.4","pandas":"2.2.2"}
for mod, want in REQUIRED.items():
    m = importlib.import_module(mod)
    got = getattr(m, "__version__", "?")
    ok  = got.startswith(want)
    print(f"{'OK ' if ok else 'WARN'} {mod:12s} want {want:8s} got {got}")

In [ ]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))
    a = torch.randn(2048, 2048, device="cuda")
    b = torch.randn(2048, 2048, device="cuda")
    c = a @ b
    torch.cuda.synchronize()
    print("GPU matmul OK, sum =", float(c.sum()))
else:
    print("WARNING: CUDA not available. Update NVIDIA driver >= 525.")

In [ ]:
from pathlib import Path
# notebook is in setup/ ; project root is one level up
proj = Path.cwd().parent / "iris_replication"
print("Project root:", proj)
assert proj.exists(), "Run setup/create_project.bat first"

expected_dirs = ["configs","environment","agent","analysis",
                 "checkpoints","logs","results","results/figures","data"]
for d in expected_dirs:
    ok = (proj/d).is_dir()
    print(f"{'OK ' if ok else 'MISSING'}  {d}/")

expected_files = ["configs/default.yaml","environment/__init__.py",
    "environment/channel.py","environment/allocation.py",
    "environment/ris_env.py","environment/quantization.py",
    "agent/__init__.py","agent/replay_buffer.py","agent/dqn.py",
    "analysis/__init__.py","analysis/says_fit.py","analysis/quantization.py",
    "analysis/fairness.py","train.py","evaluate.py","requirements.txt",
    "README.md","commands.txt","timeline.txt","work done.txt"]
for f in expected_files:
    ok = (proj/f).is_file()
    print(f"{'OK ' if ok else 'MISSING'}  {f}")

In [ ]:
import numpy as np
from scipy.special import erfc

# Eq. 14-16
b = 3
L = 2**b
Delta = np.pi / 2**b
var_eps = Delta**2 / 12
print(f"Eq.14  L(2^b)          = {L}")
print(f"Eq.15  Delta           = {Delta:.6f}")
print(f"Eq.16  sigma_eps^2     = {var_eps:.6e}")

# Eq. 17 small-M
def G(M, b):
    D = np.pi / 2**b
    Phi = np.sinc(D / (2 * np.pi))
    return 1/M + (1 - 1/M) * Phi**2
print("Eq.17  G(b=1) small-M :", G(8, 1))
print("Eq.17  G(b=6) small-M :", G(8, 6))

# Eq. 12 Jain
R = np.array([1.,2.,3.,4.])
print("Eq.12  JFI             :", (R.sum()**2)/(len(R)*(R**2).sum()))

# Eq. 13 SAYS pdf at Table 1 params, first 3 samples
def says_pdf(x, lam, mu, s, a, bb, ap, bp):
    if x <= mu:
        return (lam/2)*np.exp(lam*(2*mu + lam*s**2 - 2*a*x))*erfc((mu + lam*s**2 - bb*x)/(s*np.sqrt(2)))
    return (lam/2)*np.exp(lam*(2*mu + lam*s**2 - 2*ap*x))*erfc((mu + lam*s**2 - bp*x)/(s*np.sqrt(2)))

p = dict(lam=0.000487, mu=9.033490, s=15.116251,
         a=15.767309, bb=0.803189, ap=1.050565, bp=0.554320)
print("Eq.13  SAYS pdf f(1)   :", says_pdf(1.0, **p))
print("Eq.13  SAYS pdf f(9)   :", says_pdf(9.0, **p))
print("Eq.13  SAYS pdf f(20)  :", says_pdf(20.0, **p))
print("\nAll math checks executed.")